In [1]:
from sagemaker.workflow.pipeline_context import PipelineSession
import sagemaker

pipeline_session = PipelineSession()

sm_sess = sagemaker.Session()
role = sagemaker.get_execution_role()

sagemaker.config INFO - Fetched defaults config from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemak

In [2]:
bucket = "amazon-sagemaker-622055002283-us-east-1-b37b41a56cd8"
prefix = "dzd_4dt0rvdnr1hoiv/5vt5uv9jpcqmxz/dev/bid-predictor-2025-11-11-17-30-54/output"
model_file_name = "model"
model_data = f"s3://{bucket}/{prefix}/{model_file_name}.tar.gz"

# instance_type = "ml.g5.xlarge"
instance_type = "ml.m5.xlarge"

image_uri = "622055002283.dkr.ecr.us-east-1.amazonaws.com/bid-predictor-sklearn-inference-gpu:latest"

In [3]:
from sagemaker.workflow.model_step import ModelStep
from sagemaker.model import Model


custom_model = Model(
    model_data=model_data,
    role=role,
    image_uri=image_uri,
    sagemaker_session=pipeline_session
)

sagemaker.config INFO - Applied value from config key = SageMaker.Model.VpcConfig


In [4]:
model_step = ModelStep(
    name="CreateBidPredictorModel",
    step_args=custom_model.create(instance_type=instance_type)
)

In [5]:
output_path = "s3://amazon-sagemaker-622055002283-us-east-1-b37b41a56cd8/dzd_4dt0rvdnr1hoiv/5vt5uv9jpcqmxz/data/output"
input_path = "s3://amazon-sagemaker-622055002283-us-east-1-b37b41a56cd8/dzd_4dt0rvdnr1hoiv/5vt5uv9jpcqmxz/data/air_canada_and_lot/evaluation_sets/eval_bid_data_snapshots_v2_3_or_mode_bids.parquet"

In [6]:
from sagemaker.transformer import Transformer
from sagemaker.workflow.steps import TransformStep
from sagemaker.inputs import TransformInput

transformer = Transformer(
    model_name=model_step.properties.ModelName,
    instance_count=1,
    instance_type=instance_type,
    output_path=output_path,
    # assemble_with="Line",
    accept="application/x-parquet",
    sagemaker_session=pipeline_session
)

batch_transform_step = TransformStep(
    name="BatchInference",
    transformer=transformer,
    inputs=TransformInput(
        data=input_path,
        content_type="application/x-parquet",
        split_type="None"
    )
)

In [7]:
from sagemaker.workflow.pipeline import Pipeline

pipeline = Pipeline(
    name="BidPredictorBatchInferencePipeline",
    parameters=[],
    steps=[model_step, batch_transform_step],
    sagemaker_session=pipeline_session
)

pipeline.create()
execution = pipeline.start()

sagemaker.config INFO - Applied value from config key = SageMaker.Pipeline.RoleArn
